# The memory cap's verify-on-return: what it costs

Fork #9 (parawanderer/cpython) runs the memory cap's hook check (`mem_verify`, three probe allocations) again whenever
an approved call into C returns, so a leaf that drops the hook mid-run cannot leave the cell uncounted. #9 chose the
FULL probe over a variant that probes only when some domain's top allocator changed (`lab/mem-recheck-gate`), on native
numbers. Under wasm32 an indirect call is a larger share of a tight loop, so the trade may read differently; this
notebook keeps the rounds so it can be revisited. Ruled 2026-10-21 (repl-specs): the full probe stands, since the gate
is a proxy for "ours is in the chain"; a narrower split (the full probe only after a call that is not one of CPython's
own builtins) is Shane's to take or not.

Shapes (`shapes.py`): `trivial` (a `return 1` run: the entry, which the fix should not move), `unlifted_c` (200k `len()`
calls), `lifted_leaf` (200k calls of a lifted host leaf), `py_func` (200k calls of the cell's own def: not checked, the
control for the call path). A shape's figure for one process is the MIN over its reps; across rounds, the median of those.

In [1]:
import hashlib, json
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "python-dialect" / "verify-cost"
DATA = HERE / "data"
SHAPES = ["trivial", "unlifted_c", "lifted_leaf", "py_func"]
pd.set_option("display.width", 160)
pd.DataFrame([{"file": p.name, "bytes": p.stat().st_size, "sha256": hashlib.sha256(p.read_bytes()).hexdigest()[:16]}
              for p in sorted(DATA.iterdir())])

,file,bytes,sha256
0,native-full.jsonl,7680,70eb561c5633ad68
1,native-gate.jsonl,4260,eb66768855d183f2
2,wasm-rounds-trial.json,773197,a2f6b01a3366ed1d


## Native (macOS arm64), as in #9's body

8 interleaved rounds of 7c40b038 (`base`) against #9 (`fixed`), then 6 rounds against the changed-top variant (`gate`),
a fresh process per build per round. The ratio is taken per round, then summarised.

In [2]:
def native(path):
    return pd.DataFrame([json.loads(l) for l in open(path)])

def summarise(df, base, other, value):
    rows = []
    for s in [x for x in SHAPES if x in set(df["shape"])]:   # df.shape is the frame's dimensions
        b = df[(df["shape"] == s) & (df.build == base)].set_index("round")[value]
        o = df[(df["shape"] == s) & (df.build == other)].set_index("round")[value]
        r = (o / b).dropna()
        rows.append({"shape": s, f"{base} ns": b.median(), f"{other} ns": o.median(), "ratio median": r.median(),
                     "ratio min": r.min(), "ratio max": r.max(), "rounds": len(r)})
    return pd.DataFrame(rows).round(3)

display(summarise(native(DATA / "native-full.jsonl"), "base", "fixed", "ns_per_call_min"))
summarise(native(DATA / "native-gate.jsonl"), "base", "gate", "ns_per_call_min")

,shape,base ns,fixed ns,ratio median,ratio min,ratio max,rounds
0,trivial,73145.95,73687.95,1.008,0.965,1.020,8
1,unlifted_c,202.55,259.45,1.261,1.226,1.354,8
2,lifted_leaf,251.35,306.90,1.204,1.021,1.308,8
3,py_func,168.25,166.60,0.992,0.875,1.135,8


,shape,base ns,gate ns,ratio median,ratio min,ratio max,rounds
0,unlifted_c,219.15,245.20,1.126,1.081,1.227,6
1,lifted_leaf,268.20,301.55,1.137,1.040,1.198,6
2,py_func,183.70,187.55,1.019,0.965,1.122,6


## wasm32 (Pyodide under Node), every rep

`drive.mjs`: one fresh Pyodide per core per round, the order rotated each round. `base` and `base_aa` load the SAME
7c40b038 core: their ratio is the noise floor. `full` is #9 (in the `trial` set a core built from its branch commit
8575f359, in the `release` set its released core); `gate` is the variant built from `lab/mem-recheck-gate` with the
same recipe. Each set is summarised on its own: rounds are paired only within a set.

In [3]:
sets = {}
for n in ("release", "trial"):
    p = DATA / f"wasm-rounds-{n}.json"
    if p.exists():
        sets[n] = pd.DataFrame([{"round": r["round"], "build": r["label"], "core": r["core"], "shape": s, "ns": min(v),
                                 "reps": len(v)} for r in json.loads(p.read_text()) for s, v in r["ns"].items()])
tables = {}
for n, wasm in sets.items():
    print(f"=== {n}:", wasm.groupby("build").core.first().to_dict(), "rounds:", wasm["round"].nunique())
    for o in ["base_aa", "full", "gate"]:
        if o in set(wasm.build):
            tables[n, o] = summarise(wasm, "base", o, "ns")
            print(f"{n}: {o} / base")
            display(tables[n, o])

=== trial: {'base': 'core-7c40b038', 'base_aa': 'core-7c40b038', 'full': 'core-8575f359', 'gate': 'core-9c7e5fb8'} rounds: 8
trial: base_aa / base


,shape,base ns,base_aa ns,ratio median,ratio min,ratio max,rounds
0,trivial,153250.000,143458.000,1.002,0.866,1.027,8
1,unlifted_c,496.448,492.692,0.962,0.918,1.098,8
2,lifted_leaf,571.491,574.449,1.014,0.977,1.027,8
3,py_func,361.277,369.849,0.994,0.963,1.073,8


trial: full / base


,shape,base ns,full ns,ratio median,ratio min,ratio max,rounds
0,trivial,153250.000,149874.500,1.050,0.879,1.077,8
1,unlifted_c,496.448,692.702,1.412,1.288,1.526,8
2,lifted_leaf,571.491,751.160,1.336,1.270,3.326,8
3,py_func,361.277,383.271,1.043,1.016,1.903,8


trial: gate / base


,shape,base ns,gate ns,ratio median,ratio min,ratio max,rounds
0,trivial,153250.000,149312.500,1.040,0.895,1.072,8
1,unlifted_c,496.448,531.247,1.085,1.012,1.146,8
2,lifted_leaf,571.491,611.823,1.091,1.034,1.125,8
3,py_func,361.277,366.398,1.049,0.979,1.077,8


## The noise floor against each candidate

For each set and shape: the A/A ratio's range, and each candidate's median ratio. A candidate's median inside the A/A
range is not distinguishable from no change on these rounds.

In [4]:
rows = []
for n in sets:
    aa = tables.get((n, "base_aa"))
    for s in SHAPES:
        row = {"set": n, "shape": s}
        if aa is not None:
            a = aa.set_index("shape").loc[s]
            row["A/A min"], row["A/A max"] = a["ratio min"], a["ratio max"]
        for c in ["full", "gate"]:
            if (n, c) in tables:
                m = tables[n, c].set_index("shape").loc[s, "ratio median"]
                row[f"{c} median"] = m
                if aa is not None:
                    row[f"{c} beyond noise"] = bool(m > row["A/A max"] or m < row["A/A min"])
        rows.append(row)
pd.DataFrame(rows).round(3)

,set,shape,A/A min,A/A max,full median,full beyond noise,gate median,gate beyond noise
0,trial,trivial,0.866,1.027,1.050,True,1.040,True
1,trial,unlifted_c,0.918,1.098,1.412,True,1.085,False
2,trial,lifted_leaf,0.977,1.027,1.336,True,1.091,True
3,trial,py_func,0.963,1.073,1.043,False,1.049,False


In [5]:
# ns per approved call that each candidate adds over base, under wasm32 (median of per-round differences)
diff = []
for n, wasm in sets.items():
    for c in [x for x in ["full", "gate", "base_aa"] if x in set(wasm.build)]:
        for s in ["unlifted_c", "lifted_leaf", "py_func"]:
            b = wasm[(wasm["shape"] == s) & (wasm.build == "base")].set_index("round").ns
            o = wasm[(wasm["shape"] == s) & (wasm.build == c)].set_index("round").ns
            diff.append({"set": n, "build": c, "shape": s, "added ns/call (median)": (o - b).median()})
pd.DataFrame(diff).pivot(index=["set", "shape"], columns="build", values="added ns/call (median)").round(1)

build              base_aa   full  gate
set   shape                            
trial lifted_leaf      8.1  194.0  51.0
      py_func         -2.6   14.3  18.8
      unlifted_c     -18.9  200.7  42.0

## The trivial run's spread

A trivial run is about 150 µs under wasm32, mostly compile; its per-round ratio spreads wider than the call shapes', so
a median a few percent off 1 is not read as a cost.

In [6]:
spread = []
for n, wasm in sets.items():
    t = wasm[wasm["shape"] == "trivial"]
    for b, g in t.groupby("build"):
        spread.append({"set": n, "build": b, "min µs": g.ns.min() / 1e3, "median µs": g.ns.median() / 1e3,
                       "max µs": g.ns.max() / 1e3, "max/min": g.ns.max() / g.ns.min()})
pd.DataFrame(spread).round(2)

,set,build,min µs,median µs,max µs,max/min
0,trial,base,128.75,153.25,155.83,1.21
1,trial,base_aa,128.92,143.46,156.37,1.21
2,trial,full,135.54,149.87,163.92,1.21
3,trial,gate,135.96,149.31,163.17,1.20
